# PRISM extension - Camelyon17 hospital transfer, part 2: analysis

Part 1 produced embeddings for 8 foundation models across 5 hospitals, every
hospital carrying the same binary task and the same label definition. This
notebook runs the PRISM protocol over them.

| grid | cells |
|---|---|
| in-distribution | 5 hospitals x 8 models x 6 fractions x 3 seeds = **720** |
| transfer | 20 directed pairs x 8 models x 6 fractions x 3 seeds = **2,880** |

## What this settles

Section 4 of the paper reports that additional source-domain labels worsen
target calibration. Every pair supporting that claim also changes the label
definition, so the effect could belong to the relabelling rather than to the
shift. Here the label definition is fixed and only the acquisition conditions
change, across twenty directed pairs instead of four.

## Protocol notes

**Balanced re-split.** The stored `.npy` files were written under a
slide-disjoint split that left tumour prevalence between 0.1% and 83% across
splits, which would confound ECE with prevalence. `resplit.json` is an index
map applied at load time that gives exactly 14,000/3,000/3,000 at 50/50 in
every hospital. The `.npy` files themselves are untouched. Slide-level leakage
is structurally impossible in the transfer grid, where source and target are
different hospitals; only the in-distribution numbers are optimistic, and they
are reported as such.

**Two label budgets, one probe fit.** Reviewer tp5b observed that calling a
condition "1% labels" while fitting the temperature on the full validation
split understates what the condition consumes. Both are reported here: the
temperature is fitted once on the full validation split and once on a subset
scaled by the same fraction, from the same probe, so the extra cost is
negligible. At 1% a proportional cell uses 140 training and 30 validation
labels.

**L2 normalisation** is applied at load time so the embeddings match the main
benchmark and `C = 1.0` means the same thing on both datasets.

CPU only, roughly 2 to 4 hours. Checkpointed per (model, hospital) and per
(model, pair).

In [1]:
import os, gc, glob, json, time, warnings
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
from scipy.optimize import minimize_scalar
from scipy.stats import spearmanr, kendalltau, wilcoxon
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive')

BASE     = '/content/drive/MyDrive/PRISM'
EMB_ROOT = f'{BASE}/embeddings_camelyon17'
OUT_DIR  = f'{BASE}/results_v2'
CKPT_ID  = f'{OUT_DIR}/cam_indomain_parts'
CKPT_OOD = f'{OUT_DIR}/cam_transfer_parts'
os.makedirs(CKPT_ID, exist_ok=True)
os.makedirs(CKPT_OOD, exist_ok=True)

MODELS = ['CLIP','PLIP','CONCH','VIRCHOW2','UNI','GigaPath','H-Optimus-0','MIDNIGHT']
MKEYS  = ['clip','plip','conch','virchow2','uni','gigapath','h_optimus_0','midnight']
M2K    = dict(zip(MODELS, MKEYS))

HOSPITALS = [0, 1, 2, 3, 4]
PAIRS = [(a, b) for a in HOSPITALS for b in HOSPITALS if a != b]

FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
SEEDS     = [42, 123, 456]
N_BINS, C_DEFAULT, MAX_ITER = 15, 1.0, 1000

with open(f'{EMB_ROOT}/resplit.json') as f:
    RESPLIT = json.load(f)

print(f'{len(PAIRS)} directed pairs')
print(f'{len(MODELS)*len(HOSPITALS)*len(FRACTIONS)*len(SEEDS)} in-distribution fits')
print(f'{len(MODELS)*len(PAIRS)*len(FRACTIONS)*len(SEEDS)} transfer fits')
print('\nbalanced re-split, per hospital:')
for h in HOSPITALS:
    r = RESPLIT[str(h)]
    print(f'  hospital {h}: ' + '  '.join(f'{k}={len(v)}' for k, v in r.items()))

try:
    pre = pd.read_csv(f'{EMB_ROOT}/preprocessing_table.csv')
    print('\npreprocessing resolved from each model:')
    print(pre.to_string(index=False))
except FileNotFoundError:
    print('\npreprocessing_table.csv not found')

Mounted at /content/drive
20 directed pairs
720 in-distribution fits
2880 transfer fits

balanced re-split, per hospital:
  hospital 0: train=14000  val=3000  test=3000
  hospital 1: train=14000  val=3000  test=3000
  hospital 2: train=14000  val=3000  test=3000
  hospital 3: train=14000  val=3000  test=3000
  hospital 4: train=14000  val=3000  test=3000

preprocessing resolved from each model:
      model  resize       crop                     mean                      std  dim returns
       CLIP     224 (224, 224) (0.4815, 0.4578, 0.4082) (0.2686, 0.2613, 0.2758)  512  Tensor
       PLIP     224 (224, 224) (0.4815, 0.4578, 0.4082) (0.2686, 0.2613, 0.2758)  512  Tensor
   VIRCHOW2     224 (224, 224)    (0.485, 0.456, 0.406)    (0.229, 0.224, 0.225) 2560  Tensor
        UNI     224 (224, 224)    (0.485, 0.456, 0.406)    (0.229, 0.224, 0.225) 1024  Tensor
   GigaPath     224 (224, 224)    (0.485, 0.456, 0.406)    (0.229, 0.224, 0.225) 1536  Tensor
H-Optimus-0     256 (224, 224) (0.7072

## 1. Metrics and loading

In [2]:
def _ece_edges(conf, correct, edges):
    ece, n = 0.0, len(conf)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.sum() > 0:
            ece += m.sum() * abs(correct[m].mean() - conf[m].mean())
    return float(ece / n)

def conf_correct(proba, y):
    return proba[:, 1], (y == 1).astype(float)

def ece_fixed(proba, y, n_bins=N_BINS):
    c, k = conf_correct(proba, y)
    return _ece_edges(c, k, np.linspace(0, 1, n_bins + 1))

def ece_adaptive(proba, y, n_bins=N_BINS):
    c, k = conf_correct(proba, y)
    e = np.quantile(c, np.linspace(0, 1, n_bins + 1))
    e[0], e[-1] = 0.0, 1.0 + 1e-9
    e = np.unique(e)
    return ece_fixed(proba, y, n_bins) if len(e) < 3 else _ece_edges(c, k, e)

def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def fit_temperature(logits, y, bounds=(0.1, 10.0)):
    idx = np.arange(len(y))
    def nll(T):
        p = softmax(logits / T)
        return float(-np.log(p[idx, y] + 1e-12).mean())
    return float(minimize_scalar(nll, bounds=bounds, method='bounded').x)

def stratified_sample(labels, fraction, seed):
    np.random.seed(seed)
    idx_all = np.arange(len(labels))
    picked, forced = [], []
    for c in np.unique(labels):
        ci = idx_all[labels == c]
        exact = len(ci) * fraction
        n = max(1, int(exact))
        if exact < 1:
            forced.append(int(c))
        picked.extend(np.random.choice(ci, size=n, replace=False))
    return np.array(sorted(picked)), forced

def degeneracy(pred):
    cnt = np.bincount(pred, minlength=2)
    return float(cnt.max() / cnt.sum())


_CACHE = {}

def load_hospital(mkey, hospital, l2=True):
    """Concatenate the three stored splits, apply the balanced re-split index
    map, and L2-normalise so the scale matches the main benchmark. One
    hospital is cached at a time to bound memory."""
    key = (mkey, hospital)
    if key not in _CACHE:
        d = f'{EMB_ROOT}/{mkey}/hospital_{hospital}'
        X = np.concatenate([np.load(f'{d}/{s}_features.npy')
                            for s in ['train', 'val', 'test']])
        y = np.concatenate([np.load(f'{d}/{s}_labels.npy')
                            for s in ['train', 'val', 'test']])
        if l2:
            X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
        _CACHE.clear()
        _CACHE[key] = (X, y)
    return _CACHE[key]

def load_emb(mkey, hospital, split):
    X, y = load_hospital(mkey, hospital)
    pos = RESPLIT[str(hospital)][split]
    return X[pos], y[pos]

def load_all(mkey, hospital):
    """All three re-split partitions from a single read."""
    X, y = load_hospital(mkey, hospital)
    r = RESPLIT[str(hospital)]
    return {s: (X[r[s]], y[r[s]]) for s in ['train', 'val', 'test']}

def proba_chunked(clf, X, chunk=20000):
    return np.vstack([clf.predict_proba(np.asarray(X[i:i+chunk], dtype=np.float32))
                      for i in range(0, X.shape[0], chunk)])

def logits_chunked(clf, X, chunk=20000):
    out = []
    for i in range(0, X.shape[0], chunk):
        d = clf.decision_function(np.asarray(X[i:i+chunk], dtype=np.float32))
        if d.ndim == 1:
            d = d.reshape(-1, 1); d = np.hstack([-d, d])
        out.append(d)
    return np.vstack(out)

# quick check that loading works and the balance survived
_p = load_all('uni', 0)
for s, (X, y) in _p.items():
    print(f'  uni h0 {s:<6} {X.shape}  tumour {y.mean():.4f}  '
          f'|x| {np.linalg.norm(X[0]):.4f}')
del _p, _CACHE
_CACHE = {}
print('\nready')

  uni h0 train  (14000, 1024)  tumour 0.5000  |x| 1.0000
  uni h0 val    (3000, 1024)  tumour 0.5000  |x| 1.0000
  uni h0 test   (3000, 1024)  tumour 0.5000  |x| 1.0000

ready


## 2. In-distribution, per hospital

Needed on its own account and as the denominator of OOD_Stability. Because
every hospital carries the same task, this is also the first place in the
benchmark where a cross-model comparison is free of the preprocessing
heterogeneity discussed in Appendix A: all eight models were embedded here
under their own documented transform, tabulated in part 1.

The temperature is fitted twice from the same probe: once on the full
validation split and once on a subset scaled by the label fraction.

In [3]:
def run_indomain(model, hospital):
    mk = M2K[model]
    P = load_all(mk, hospital)
    Xtr, ytr = P['train']
    Xva, yva = P['val']
    Xte, yte = P['test']
    rows = []

    for frac in FRACTIONS:
        for seed in SEEDS:
            idx, forced = stratified_sample(ytr, frac, seed)
            clf = LogisticRegression(max_iter=MAX_ITER, C=C_DEFAULT,
                                     random_state=seed).fit(
                  np.asarray(Xtr[idx], dtype=np.float32), ytr[idx])

            proba = proba_chunked(clf, Xte)
            pred  = proba.argmax(1)
            try:
                auroc = roc_auc_score(yte, proba[:, 1])
            except Exception:
                auroc = np.nan

            Lva, Lte = logits_chunked(clf, Xva), logits_chunked(clf, Xte)

            T_full  = fit_temperature(Lva, yva)
            sp_full = softmax(Lte / T_full)

            vidx, _ = stratified_sample(yva, frac, seed)
            T_prop  = fit_temperature(Lva[vidx], yva[vidx])
            sp_prop = softmax(Lte / T_prop)

            rows.append(dict(
                model=model, hospital=hospital, fraction=frac, seed=seed,
                n_train=len(idx), n_val_full=len(yva), n_val_prop=len(vidx),
                auroc=auroc,
                f1_macro=f1_score(yte, pred, average='macro', zero_division=0),
                brier=brier_score_loss(yte, proba[:, 1]),
                ece_fixed=ece_fixed(proba, yte),
                ece_adaptive=ece_adaptive(proba, yte),
                temperature=T_full,
                ece_scaled_fixed=ece_fixed(sp_full, yte),
                ece_scaled_adaptive=ece_adaptive(sp_full, yte),
                temperature_prop=T_prop,
                ece_scaled_prop=ece_fixed(sp_prop, yte),
                degeneracy_share=degeneracy(pred),
                forced_classes=len(forced)))
            del proba, sp_full, sp_prop, Lva, Lte, clf
            gc.collect()

    df = pd.DataFrame(rows)
    df['degenerate'] = df['degeneracy_share'] > 0.99
    return df


t0 = time.time()
for h in HOSPITALS:
    for m in MODELS:
        out = f'{CKPT_ID}/{M2K[m]}__h{h}.csv'
        if os.path.exists(out):
            continue
        try:
            df = run_indomain(m, h)
            df.to_csv(out, index=False)
            a = df.groupby('fraction')['auroc'].mean()
            e = df.groupby('fraction')['ece_fixed'].mean()
            print(f'{m:>12} h{h}  AUROC {a.loc[0.01]:.4f}->{a.loc[1.00]:.4f}  '
                  f'ECE {e.loc[0.01]:.4f}->{e.loc[1.00]:.4f}  '
                  f'({time.time()-t0:.0f}s)')
        except Exception as e:
            print(f'{m:>12} h{h}  FAILED: {type(e).__name__}: {e}')
        gc.collect()

parts = sorted(glob.glob(f'{CKPT_ID}/*.csv'))
cam_id = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
cam_id.to_csv(f'{OUT_DIR}/camelyon17_indomain.csv', index=False)
print(f'\n{len(parts)}/40 cells, {len(cam_id)} runs -> camelyon17_indomain.csv')

        CLIP h0  AUROC 0.9396->0.9817  ECE 0.2637->0.0515  (13s)
        PLIP h0  AUROC 0.9860->0.9965  ECE 0.2345->0.0302  (25s)
       CONCH h0  AUROC 0.9913->0.9968  ECE 0.1478->0.0125  (35s)
    VIRCHOW2 h0  AUROC 0.9951->0.9988  ECE 0.2039->0.0170  (64s)
         UNI h0  AUROC 0.9967->0.9987  ECE 0.1979->0.0138  (71s)
    GigaPath h0  AUROC 0.9970->0.9988  ECE 0.1852->0.0129  (88s)
 H-Optimus-0 h0  AUROC 0.9966->0.9993  ECE 0.2073->0.0166  (105s)
    MIDNIGHT h0  AUROC 0.9949->0.9990  ECE 0.1340->0.0140  (120s)
        CLIP h1  AUROC 0.9399->0.9768  ECE 0.2472->0.0533  (131s)
        PLIP h1  AUROC 0.9693->0.9919  ECE 0.2257->0.0374  (142s)
       CONCH h1  AUROC 0.9901->0.9970  ECE 0.1764->0.0228  (150s)
    VIRCHOW2 h1  AUROC 0.9954->0.9987  ECE 0.2165->0.0196  (171s)
         UNI h1  AUROC 0.9963->0.9990  ECE 0.2031->0.0148  (185s)
    GigaPath h1  AUROC 0.9959->0.9988  ECE 0.1857->0.0168  (203s)
 H-Optimus-0 h1  AUROC 0.9968->0.9993  ECE 0.2139->0.0182  (220s)
    MIDNIGHT h1 

## 3. Transfer across all 20 directed hospital pairs

Temperature fitted on the source hospital's validation split. No target label
enters the fit at any point, and source and target are disjoint at the level of
hospital, patient and slide.

In [4]:
def run_transfer(model, src, tgt):
    mk = M2K[model]
    S = load_all(mk, src)
    Xs_tr, ys_tr = S['train']
    Xs_va, ys_va = S['val']
    Xs_tr = np.array(Xs_tr, copy=True); Xs_va = np.array(Xs_va, copy=True)
    T = load_all(mk, tgt)
    Xt_te, yt_te = T['test']
    rows = []

    for frac in FRACTIONS:
        for seed in SEEDS:
            idx, _ = stratified_sample(ys_tr, frac, seed)
            clf = LogisticRegression(max_iter=MAX_ITER, C=C_DEFAULT,
                                     random_state=seed).fit(
                  np.asarray(Xs_tr[idx], dtype=np.float32), ys_tr[idx])

            proba = proba_chunked(clf, Xt_te)
            pred  = proba.argmax(1)
            try:
                auroc = roc_auc_score(yt_te, proba[:, 1])
            except Exception:
                auroc = np.nan

            Lva, Lte = logits_chunked(clf, Xs_va), logits_chunked(clf, Xt_te)

            T_full  = fit_temperature(Lva, ys_va)
            sp_full = softmax(Lte / T_full)

            vidx, _ = stratified_sample(ys_va, frac, seed)
            T_prop  = fit_temperature(Lva[vidx], ys_va[vidx])
            sp_prop = softmax(Lte / T_prop)

            rows.append(dict(
                model=model, src=src, tgt=tgt, pair=f'h{src}->h{tgt}',
                fraction=frac, seed=seed, n_train=len(idx),
                n_val_full=len(ys_va), n_val_prop=len(vidx), auroc=auroc,
                f1_macro=f1_score(yt_te, pred, average='macro', zero_division=0),
                brier=brier_score_loss(yt_te, proba[:, 1]),
                ece_fixed=ece_fixed(proba, yt_te),
                ece_adaptive=ece_adaptive(proba, yt_te),
                temperature_src=T_full,
                ece_scaled_fixed=ece_fixed(sp_full, yt_te),
                ece_scaled_adaptive=ece_adaptive(sp_full, yt_te),
                temperature_prop=T_prop,
                ece_scaled_prop=ece_fixed(sp_prop, yt_te),
                degeneracy_share=degeneracy(pred)))
            del proba, sp_full, sp_prop, Lva, Lte, clf
            gc.collect()

    del Xs_tr, Xs_va, Xt_te, S, T
    gc.collect()
    df = pd.DataFrame(rows)
    df['degenerate'] = df['degeneracy_share'] > 0.99
    return df


t0, done = time.time(), 0
for m in MODELS:
    for src, tgt in PAIRS:
        out = f'{CKPT_OOD}/{M2K[m]}__h{src}_to_h{tgt}.csv'
        if os.path.exists(out):
            continue
        try:
            df = run_transfer(m, src, tgt)
            df.to_csv(out, index=False)
            done += 1
            s = df.groupby('fraction')['ece_fixed'].mean()
            a = df.groupby('fraction')['auroc'].mean()
            arrow = 'RISES' if s.loc[1.00] > s.loc[0.01] else 'falls'
            print(f'{m:>12} h{src}->h{tgt}  '
                  f'AUROC {a.loc[1.00]:.3f}  '
                  f'ECE {s.loc[0.01]:.3f}->{s.loc[1.00]:.3f} {arrow}  '
                  f'({time.time()-t0:.0f}s)')
        except Exception as e:
            print(f'{m:>12} h{src}->h{tgt}  FAILED: {type(e).__name__}: {e}')
        gc.collect()

parts = sorted(glob.glob(f'{CKPT_OOD}/*.csv'))
cam_ood = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
cam_ood.to_csv(f'{OUT_DIR}/camelyon17_transfer.csv', index=False)
print(f'\n{len(parts)}/160 cells, {len(cam_ood)} runs -> camelyon17_transfer.csv')

        CLIP h0->h1  AUROC 0.957  ECE 0.264->0.078 falls  (8s)
        CLIP h0->h2  AUROC 0.966  ECE 0.104->0.103 falls  (18s)
        CLIP h0->h3  AUROC 0.958  ECE 0.259->0.061 falls  (26s)
        CLIP h0->h4  AUROC 0.939  ECE 0.215->0.361 RISES  (35s)
        CLIP h1->h0  AUROC 0.949  ECE 0.201->0.055 falls  (43s)
        CLIP h1->h2  AUROC 0.817  ECE 0.121->0.041 falls  (51s)
        CLIP h1->h3  AUROC 0.967  ECE 0.272->0.061 falls  (60s)
        CLIP h1->h4  AUROC 0.670  ECE 0.100->0.281 RISES  (68s)
        CLIP h2->h0  AUROC 0.940  ECE 0.132->0.092 falls  (79s)
        CLIP h2->h1  AUROC 0.829  ECE 0.055->0.144 RISES  (89s)
        CLIP h2->h3  AUROC 0.883  ECE 0.100->0.105 RISES  (99s)
        CLIP h2->h4  AUROC 0.967  ECE 0.131->0.119 falls  (108s)
        CLIP h3->h0  AUROC 0.922  ECE 0.240->0.094 falls  (116s)
        CLIP h3->h1  AUROC 0.950  ECE 0.270->0.238 falls  (123s)
        CLIP h3->h2  AUROC 0.829  ECE 0.123->0.232 RISES  (131s)
        CLIP h3->h4  AUROC 0.794  ECE

## 4. The test: does reverse OOD scaling survive clean covariate shift?

With 20 directed pairs and 8 models there are 160 (model, pair) combinations,
so this can be answered as a rate with a significance test rather than as an
anecdote.

In [5]:
def trend_table(df, metric='ece_fixed'):
    rows = []
    for m in df['model'].unique():
        for p in df['pair'].unique():
            s = (df[(df.model == m) & (df.pair == p)]
                 .groupby('fraction')[metric].mean().sort_index())
            if len(s) < 2:
                continue
            rows.append(dict(model=m, pair=p, lo=s.iloc[0], hi=s.iloc[-1],
                             delta=s.iloc[-1] - s.iloc[0],
                             rises=bool(s.iloc[-1] > s.iloc[0]),
                             monotone=bool(np.all(np.diff(s.values) >= -1e-9))))
    return pd.DataFrame(rows)


tr = trend_table(cam_ood)
tr.to_csv(f'{OUT_DIR}/camelyon17_ece_trend.csv', index=False)
n = len(tr)

print('=== raw target ECE, 1% against 100% source labels ===\n')
print(f'  (model, pair) combinations : {n}')
print(f'  ECE rises                  : {tr.rises.sum()} of {n} '
      f'({100*tr.rises.mean():.0f}%)')
print(f'  monotone in the fraction   : {tr.monotone.sum()} of {n} '
      f'({100*tr.monotone.mean():.0f}%)')
print(f'  mean change                : {tr.delta.mean():+.4f}')
print(f'  median change              : {tr.delta.median():+.4f}')

st, pv = wilcoxon(tr['hi'], tr['lo'])
print(f'\n  Wilcoxon signed-rank, 1% against 100%: W={st:.0f}, p={pv:.2e}')

print('\n=== per model ===')
print(tr.groupby('model').agg(
    rises=('rises','sum'), of=('rises','size'),
    mean_delta=('delta','mean'), monotone=('monotone','sum')
).round(4).to_string())

CLAIM = ['UNI','VIRCHOW2','GigaPath','H-Optimus-0']
sub = tr[tr.model.isin(CLAIM)]
print(f'\n=== the four models the Section 4 claim covers ===')
print(f'  rises in {sub.rises.sum()} of {len(sub)}, '
      f'mean change {sub.delta.mean():+.4f}')

print('\n=== per pair, across models ===')
print(tr.groupby('pair').agg(
    rises=('rises','sum'), mean_delta=('delta','mean')).round(4).to_string())

print('\n=== full curves, four claim models, averaged over pairs ===')
print(cam_ood[cam_ood.model.isin(CLAIM)]
      .pivot_table(index='model', columns='fraction', values='ece_fixed')
      .round(4).to_string())

print('\n=== same, under the proportional label budget ===')
tr_prop = trend_table(cam_ood, metric='ece_scaled_prop')
print(f'  scaled ECE rises in {tr_prop.rises.sum()} of {len(tr_prop)}, '
      f'mean change {tr_prop.delta.mean():+.4f}')

=== raw target ECE, 1% against 100% source labels ===

  (model, pair) combinations : 160
  ECE rises                  : 10 of 160 (6%)
  monotone in the fraction   : 5 of 160 (3%)
  mean change                : -0.1492
  median change              : -0.1760

  Wilcoxon signed-rank, 1% against 100%: W=537, p=8.59e-24

=== per model ===
             rises  of  mean_delta  monotone
model                                       
CLIP             8  20      0.0008         5
CONCH            0  20     -0.1450         0
GigaPath         1  20     -0.1696         0
H-Optimus-0      0  20     -0.1762         0
MIDNIGHT         0  20     -0.1517         0
PLIP             1  20     -0.1635         0
UNI              0  20     -0.1966         0
VIRCHOW2         0  20     -0.1921         0

=== the four models the Section 4 claim covers ===
  rises in 1 of 80, mean change -0.1836

=== per pair, across models ===
        rises  mean_delta
pair                     
h0->h1      0     -0.1920
h0->h2   

## 5. The two label budgets

Raw ECE never touches the validation split, so it is identical under both
budgets by construction. What can differ is the temperature and anything
derived from it.

In [6]:
acc = []
for h in HOSPITALS:
    r = RESPLIT[str(h)]
    for f in FRACTIONS:
        n_tr = max(1, int(len(r['train']) * f / 2)) * 2
        n_vp = max(1, int(len(r['val'])   * f / 2)) * 2
        acc.append(dict(hospital=h, fraction=f, n_train=n_tr,
                        val_full=len(r['val']), val_prop=n_vp,
                        total_full=n_tr + len(r['val']),
                        total_prop=n_tr + n_vp,
                        val_share_full=len(r['val']) / (n_tr + len(r['val']))))
acc = pd.DataFrame(acc)
acc.to_csv(f'{OUT_DIR}/camelyon17_label_budget.csv', index=False)
print('=== realised label budget ===\n')
print(acc[acc.hospital == 0][['fraction','n_train','val_full','val_prop',
                              'total_full','total_prop','val_share_full']]
      .round(3).to_string(index=False))
print('\n(identical across hospitals: every one has 14,000 train and 3,000 val)')

print('\n=== transfer: full against proportional validation ===')
cmp = pd.DataFrame({
    'ece_raw':        cam_ood.groupby('fraction')['ece_fixed'].mean(),
    'scaled_full':    cam_ood.groupby('fraction')['ece_scaled_fixed'].mean(),
    'scaled_prop':    cam_ood.groupby('fraction')['ece_scaled_prop'].mean(),
    'T_full':         cam_ood.groupby('fraction')['temperature_src'].mean(),
    'T_prop':         cam_ood.groupby('fraction')['temperature_prop'].mean(),
})
cmp['scaled_delta'] = cmp['scaled_prop'] - cmp['scaled_full']
print(cmp.round(4).to_string())

print('\n=== how often does the temperature hit a search bound? ===')
for col in ['temperature_src', 'temperature_prop']:
    hit = ((cam_ood[col] <= 0.101) | (cam_ood[col] >= 9.999))
    print(f'  {col:<18} '
          f'{hit.groupby(cam_ood.fraction).mean().round(3).to_dict()}')

print('\n=== does post-hoc scaling help or hurt under transfer? ===')
for col, lab in [('ece_scaled_fixed','full validation'),
                 ('ece_scaled_prop','proportional')]:
    worse = (cam_ood[col] > cam_ood['ece_fixed']).mean()
    print(f'  {lab:<18} scaling worsens ECE in {100*worse:.0f}% of cells')

=== realised label budget ===

 fraction  n_train  val_full  val_prop  total_full  total_prop  val_share_full
     0.01      140      3000        30        3140         170           0.955
     0.05      700      3000       150        3700         850           0.811
     0.10     1400      3000       300        4400        1700           0.682
     0.25     3500      3000       750        6500        4250           0.462
     0.50     7000      3000      1500       10000        8500           0.300
     1.00    14000      3000      3000       17000       17000           0.176

(identical across hospitals: every one has 14,000 train and 3,000 val)

=== transfer: full against proportional validation ===
          ece_raw  scaled_full  scaled_prop  T_full  T_prop  scaled_delta
fraction                                                                 
0.01       0.2485       0.1224       0.1220  0.5004  0.3080       -0.0004
0.05       0.1735       0.1089       0.1089  0.8265  0.7439       

## 6. Side by side with the label-definition-shift pairs

The comparison that decides how Section 4 should be worded.

In [7]:
old = pd.read_csv(f'{OUT_DIR}/ood_all_v2.csv')
old_tr = trend_table(old)

print('=== reverse scaling rate, two transfer designs ===\n')
print(f"{'design':<46}{'rises':>7}{'of':>5}{'rate':>8}{'mean delta':>12}")
print('-' * 78)
for label, t in [('label-definition shift, 4 pairs (submitted)', old_tr),
                 ('covariate shift only, 20 hospital pairs', tr)]:
    print(f'{label:<46}{t.rises.sum():>7}{len(t):>5}'
          f'{100*t.rises.mean():>7.0f}%{t.delta.mean():>12.4f}')

print('\n=== the four claim models in each design ===')
for label, t in [('label-definition shift', old_tr), ('covariate shift', tr)]:
    s = t[t.model.isin(CLAIM)]
    print(f'  {label:<24} {s.rises.sum()}/{len(s)}, '
          f'mean delta {s.delta.mean():+.4f}')

print('\n=== magnitudes at full supervision ===')
cmp2 = pd.DataFrame({
    'label_shift_ECE':   old[old.fraction == 1.0].groupby('model')['ece_fixed'].mean(),
    'covariate_ECE':     cam_ood[cam_ood.fraction == 1.0].groupby('model')['ece_fixed'].mean(),
    'label_shift_AUROC': old[old.fraction == 1.0].groupby('model')['auroc'].mean(),
    'covariate_AUROC':   cam_ood[cam_ood.fraction == 1.0].groupby('model')['auroc'].mean(),
})
print(cmp2.round(4).to_string())
print('\nAUROC well above 0.5 in the covariate column confirms the transfer is')
print('meaningful, which the label-definition pairs could not guarantee.')

print('\n=== degeneracy, two designs ===')
print(f'  label-definition pairs: {old.degenerate.mean():.3f} of cells collapse')
print(f'  hospital pairs        : {cam_ood.degenerate.mean():.3f} of cells collapse')

print('\n=== does the model ranking agree across designs? ===')
for metric in ['auroc', 'ece_fixed']:
    a = old[old.fraction == 1.0].groupby('model')[metric].mean()
    b = cam_ood[cam_ood.fraction == 1.0].groupby('model')[metric].mean()
    k = a.index.intersection(b.index)
    print(f'  {metric:<12} Kendall tau = {kendalltau(a[k], b[k]).correlation:+.3f}')

=== reverse scaling rate, two transfer designs ===

design                                          rises   of    rate  mean delta
------------------------------------------------------------------------------
label-definition shift, 4 pairs (submitted)        28   32     88%      0.0820
covariate shift only, 20 hospital pairs            10  160      6%     -0.1492

=== the four claim models in each design ===
  label-definition shift   14/16, mean delta +0.0909
  covariate shift          1/80, mean delta -0.1836

=== magnitudes at full supervision ===
             label_shift_ECE  covariate_ECE  label_shift_AUROC  covariate_AUROC
model                                                                          
CLIP                  0.2668         0.1751             0.5748           0.8845
CONCH                 0.3011         0.0477             0.4993           0.9840
GigaPath              0.3152         0.1345             0.5699           0.9922
H-Optimus-0           0.3247         0.13

## 7. What 20 pairs makes possible

Two analyses the four-pair design could not support: whether transfer
difficulty belongs to the target hospital or to the pair, and OOD_Stability
computed on pairs that actually share a label definition.

In [8]:
full = cam_ood[cam_ood.fraction == 1.0]

print('=== AUROC by source and target hospital, averaged over models ===\n')
mat = full.pivot_table(index='src', columns='tgt', values='auroc')
print(mat.round(3).to_string())
row_sd = mat.mean(axis=1).std()
col_sd = mat.mean(axis=0).std()
print(f'\n  spread of source means : {row_sd:.4f}')
print(f'  spread of target means : {col_sd:.4f}')
print(f'  -> difficulty belongs mainly to the '
      f'{"target" if col_sd > row_sd else "source"} hospital')

print('\n=== ECE by source and target hospital ===\n')
print(full.pivot_table(index='src', columns='tgt', values='ece_fixed')
      .round(3).to_string())

id100 = cam_id[cam_id.fraction == 1.0].groupby(['model','hospital'])['auroc'].mean()
rows = []
for m in MODELS:
    ratios, drops = [], []
    for src, tgt in PAIRS:
        o = full[(full.model == m) & (full.src == src) &
                 (full.tgt == tgt)]['auroc'].mean()
        try:
            i = id100.loc[(m, src)]
        except KeyError:
            continue
        if np.isfinite(o) and i:
            ratios.append(min(o / i, 1.0))
            drops.append(i - o)
    if ratios:
        rows.append(dict(
            model=m,
            id_auroc=id100.loc[m].mean() if m in id100.index.get_level_values(0) else np.nan,
            ood_auroc=full[full.model == m]['auroc'].mean(),
            drop_mean=np.mean(drops), drop_worst=np.max(drops),
            ood_stability=np.mean(ratios)))
comp = pd.DataFrame(rows).set_index('model')

cri = {}
for m in MODELS:
    s = cam_id[(cam_id.model == m) & (cam_id.fraction == 1.0)]
    if s.empty or m not in comp.index:
        continue
    a = s.groupby('hospital')['auroc'].mean().mean()
    e = float(np.clip(s.groupby('hospital')['ece_scaled_fixed'].mean().mean(), 0, 1))
    cri[m] = a * (1 - e) * comp.loc[m, 'ood_stability']
comp['cri'] = pd.Series(cri)
comp.to_csv(f'{OUT_DIR}/camelyon17_cri_components.csv')

print('\n=== CRI components on same-task transfer ===\n')
print(comp.round(4).sort_values('ood_stability', ascending=False).to_string())

print('\n=== component rankings against the composite ===')
rk = pd.DataFrame({
    'by_id_auroc':  comp['id_auroc'].rank(ascending=False),
    'by_ood_auroc': comp['ood_auroc'].rank(ascending=False),
    'by_drop':      comp['drop_mean'].rank(ascending=True),
    'by_cri':       comp['cri'].rank(ascending=False)}).astype(int)
print(rk.to_string())
for c in ['by_id_auroc','by_ood_auroc','by_drop']:
    t_ = kendalltau(rk[c], rk['by_cri']).correlation
    print(f'  {c:<14} vs by_cri: {t_:+.3f}')

print('\n=== decoupling on same-task transfer ===')
def rho_at(fraction, ece_col='ece_scaled_fixed'):
    out = []
    for p in cam_ood['pair'].unique():
        s = (cam_ood[(cam_ood.pair == p) & (cam_ood.fraction == fraction)]
             .groupby('model')[['auroc', ece_col]].mean().dropna())
        if len(s) < 3:
            continue
        out += list(zip(s['auroc'].rank(ascending=False).values,
                        s[ece_col].rank(ascending=True).values))
    if len(out) < 4:
        return np.nan
    x, y = zip(*out)
    return spearmanr(x, y).correlation

print(f"{'fraction':>10}{'rho':>10}")
for f in FRACTIONS:
    print(f'{f:>10.2f}{rho_at(f):>10.3f}')

=== AUROC by source and target hospital, averaged over models ===

tgt      0      1      2      3      4
src                                   
0      NaN  0.986  0.992  0.988  0.985
1    0.989    NaN  0.967  0.990  0.934
2    0.989  0.961    NaN  0.973  0.990
3    0.985  0.986  0.966    NaN  0.960
4    0.983  0.935  0.992  0.964    NaN

  spread of source means : 0.0077
  spread of target means : 0.0084
  -> difficulty belongs mainly to the target hospital

=== ECE by source and target hospital ===

tgt      0      1      2      3      4
src                                   
0      NaN  0.059  0.067  0.049  0.166
1    0.056    NaN  0.085  0.049  0.182
2    0.053  0.096    NaN  0.069  0.095
3    0.054  0.081  0.123    NaN  0.246
4    0.098  0.151  0.089  0.116    NaN

=== CRI components on same-task transfer ===

             id_auroc  ood_auroc  drop_mean  drop_worst  ood_stability     cri
model                                                                         
VIRCHOW2       

## 8. Reading the result

**If ECE rises in most of the 160 combinations**, the reverse-scaling finding
belongs to covariate shift and not to the relabelling. Section 4 can then state
it without qualification, and the four original pairs become a secondary
setting. This is the outcome that answers Reviewer tp5b's objection at its root
rather than at its wording, and it is what would make the higher-tier journals
viable.

**If it rises in roughly half**, the effect is real but pair-dependent, which
is what the four-pair result already suggested. Report the rate, the Wilcoxon
test and the per-pair table, and let the number stand.

**If it does not rise**, the finding is specific to label-definition shift.
That is a narrower claim and a more accurate one. It would also be the first
evidence that the two kinds of shift behave differently, which is worth a
section of its own rather than a retraction.

Either way this dataset now carries a stated preprocessing protocol, a balanced
split, an explicit label budget and 20 transfer pairs with fixed label
semantics. Four of the limitations raised against the submitted version are
answered by construction here.